# Stochastic Gradient, Momentum, and Nesterov Acceleration

- Module: 01 Optimization
- Topic: stochastic and momentum methods
- Estimated runtime: 10 minutes
- Prerequisites: gradient descent, finite-sum objectives, basic probability intuition
- Expected outputs: trajectory comparisons on a noisy empirical risk surface, convergence summaries, and a momentum overshoot experiment

## Learning goals

- Compare deterministic and stochastic updates on the same finite-sum objective.
- See how momentum smooths noisy gradients and speeds travel through flat regions.
- Observe that too much momentum can turn acceleration into oscillation.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().parents[2]
src_dir = repo_root / "modules" / "01-optimization" / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from optim_utils import (
    adagrad,
    adam,
    gradient_descent,
    make_quadratic,
    make_shifted_least_squares,
    momentum_descent,
    newton_method,
    plot_convergence,
    plot_trajectories,
    rmsprop,
    rosenbrock_objective,
    saddle_objective,
    stochastic_gradient_descent,
    summarize_history,
)

np.set_printoptions(precision=4, suppress=True)


## Finite-sum objective with visible noise

We build an empirical risk from several shifted quadratic samples. The average objective is still convex with a clear minimizer, but individual sample gradients point in slightly different directions, which creates stochastic noise.

In [ ]:
centers = np.array([
    [1.5, 0.5],
    [1.2, -0.2],
    [0.7, 0.3],
    [1.0, 0.8],
    [1.4, -0.4],
    [0.9, 0.1],
], dtype=float)
objective = make_shifted_least_squares(centers, name="empirical_risk_quadratic")
start = np.array([-2.0, -1.8], dtype=float)
objective.minimizer

## Run four optimizers from the same starting point

The deterministic method sees the full gradient each step. SGD samples one term at a time, while momentum and Nesterov keep a velocity state that reuses directional information from earlier iterations.

In [ ]:
gd_history = gradient_descent(objective, start=start, learning_rate=0.35, steps=30)
gd_history["name"] = "GD"

sgd_history = stochastic_gradient_descent(objective, start=start, learning_rate=0.18, steps=60, seed=7)
sgd_history["name"] = "SGD"

momentum_history = momentum_descent(objective, start=start, learning_rate=0.16, momentum=0.82, steps=30)
momentum_history["name"] = "Momentum"

nesterov_history = momentum_descent(objective, start=start, learning_rate=0.14, momentum=0.82, steps=30, nesterov=True)
nesterov_history["name"] = "Nesterov"

[summarize_history(history, objective) | {"name": history["name"]} for history in [gd_history, sgd_history, momentum_history, nesterov_history]]

## Trajectory comparison

SGD jitters because each step follows one sample gradient. Momentum-based methods smooth the path and can move farther along the valley once the velocity aligns with the useful direction.

In [ ]:
fig, _ = plot_trajectories(
    objective,
    [gd_history, sgd_history, momentum_history, nesterov_history],
    x_range=(-2.5, 2.5),
    y_range=(-2.5, 1.8),
    title="Deterministic, stochastic, and momentum methods",
)
plt.show()
plt.close(fig)

## Compare objective decrease and gradient norms

The stochastic curve is noisier because the update direction is only an unbiased estimate of the full gradient. Momentum and Nesterov typically reduce the objective faster than plain GD on this surface.

In [ ]:
fig, _ = plot_convergence(
    [gd_history, sgd_history, momentum_history, nesterov_history],
    title="Finite-sum optimizer comparison",
)
plt.show()
plt.close(fig)

## Inspect the stochastic noise directly

A short look at the sampled indices helps explain why the SGD path shakes around the minimizer instead of settling into a smooth line.

In [ ]:
sgd_history["sample_indices"][:15]

## What happens when momentum is too aggressive?

Momentum helps when the velocity estimate carries useful signal forward. If the coefficient is too close to one relative to the step size and curvature, the method overshoots and oscillates around the minimizer.

In [ ]:
overshoot_history = momentum_descent(objective, start=start, learning_rate=0.32, momentum=0.96, steps=24)
overshoot_history["name"] = "Momentum overshoot"
fig, _ = plot_trajectories(
    objective,
    [momentum_history, overshoot_history],
    x_range=(-2.5, 2.5),
    y_range=(-2.5, 1.8),
    title="Helpful momentum versus overshoot",
)
plt.show()
plt.close(fig)

fig, _ = plot_convergence([momentum_history, overshoot_history], title="Momentum failure mode")
plt.show()
plt.close(fig)

{
    "stable_momentum_final_value": summarize_history(momentum_history, objective)["final_value"],
    "overshoot_final_value": summarize_history(overshoot_history, objective)["final_value"],
}

## Summary

Stochasticity trades exact descent for cheap, noisy updates. Momentum and Nesterov improve the geometry of first-order search by keeping directional memory, but the same memory can create ringing when the hyperparameters are too aggressive.

## Next steps

- Increase the number of sample centers and compare how the SGD path changes.
- Tune the momentum coefficient while keeping the learning rate fixed.
- Compare these first-order methods with Newton updates on a curved valley.